# Setup

In [ ]:
# Colab-specific setup
import pathlib

if 'google.colab' not in str(get_ipython()):
    base_folder = pathlib.Path('../../')
else:
    # install extra notebook dependencies in Colab
    ! uv pip install 'watermark==2.4.*' 'pyyaml==6.0.*'

    # mount colab folder
    #from google.colab import drive
    #drive.mount('/content/drive')
    #base_folder = pathlib.Path('/content/drive/MyDrive/Vision/')

    # directly download datasets
    ! mkdir -p 'prepared'
    ! wget -nv -P 'datasets' https://raw.githubusercontent.com/mgmalheiros/vision/master/datasets/1-coins-small-image.zip
    ! wget -nv -P 'datasets' https://raw.githubusercontent.com/mgmalheiros/vision/master/datasets/1-coins-small-annot.zip
    base_folder = pathlib.Path('./')

In [ ]:
# show library versions
import watermark

# scikit-image also installs imageio and pillow
print(watermark.watermark(packages='skimage,imageio,PIL,pandas,yaml'))

# Image conversion
1. read every image from the source dataset
2. convert images to gray
3. save images to a **prepared** dataset, in a **deterministic ZIP** file (same convention as `0-template.ipynb`)

In [ ]:
import io
import json
import zipfile

import imageio.v3 as iio
import matplotlib.pyplot as plt
import pandas as pd
import skimage
import yaml

In [ ]:
# 'small' = the 100-image demo set tracked in the repo (this is what Colab downloads too).
# 'full'  = the 6021-image "regression" dataset from IEEE Dataport
#           (https://ieee-dataport.org/open-access/brazilian-coin-detection-dataset).
#           Download it, run util/prepare-full-dataset.py to reorganize it into
#           datasets/1-coins-full-{image,annot}.zip (same layout as the small ZIPs),
#           then switch this to 'full' and re-run this notebook. Both
#           datasets/1-coins-full-*.zip and prepared/1-coins-full-* are gitignored and
#           not fetched by the Colab branch above, so a 'full' run stays local only.
dataset_size = 'full'  # 'small' or 'full'

date_time = (1980, 1, 1, 0, 0, 0) # ensure all zip entries have the same time

images_zip = base_folder / 'datasets' / f'1-coins-{dataset_size}-image.zip'
annot_zip  = base_folder / 'datasets' / f'1-coins-{dataset_size}-annot.zip'

prepared_images   = base_folder / 'prepared' / f'1-coins-{dataset_size}-image.zip'
prepared_labels   = base_folder / 'prepared' / f'1-coins-{dataset_size}-labels.csv'
prepared_metadata = base_folder / 'prepared' / f'1-coins-{dataset_size}-metadata.yaml'

In [ ]:
with zipfile.ZipFile(images_zip, 'r') as src, \
     zipfile.ZipFile(prepared_images, 'w') as dst:
    namelist = sorted(n for n in src.namelist() if n.endswith('.jpg'))

    for input_name in namelist: # already sorted, so zip entries stay deterministic
        data = io.BytesIO(src.read(input_name))
        image = skimage.io.imread(data, as_gray=True)
        image = skimage.util.img_as_ubyte(image)

        output_name = pathlib.Path(input_name).with_suffix('.jpg').name

        data = io.BytesIO()
        iio.imwrite(data, image, extension='.jpg')
        dst.writestr(zipfile.ZipInfo(output_name, date_time=date_time), data.getbuffer())

print(f'wrote {len(namelist)} images to {prepared_images}')

# Label consolidation
Each image has a matching `labelme` annotation (one JSON per image). A shape's `label` is a coin
denomination (`5`, `10`, `25`, `50`, `100`) except for the occasional `finger` shape, which marks
something that was caught in frame by accident and is not a coin.

For every image this produces one row in the **labels** CSV:

| column | meaning |
| --- | --- |
| `name` | image file name, without extension |
| `labels` | every shape's raw label, comma-joined (includes `finger`, if present) |
| `real_count` | ground-truth **coin** count — `finger` shapes excluded |

`real_count` is what every `2-processing-*.ipynb` notebook compares its detections against.

In [ ]:
names, labels_col, counts = [], [], []

with zipfile.ZipFile(annot_zip, 'r') as archive:
    for entry in sorted(n for n in archive.namelist() if n.endswith('.json')):
        annotation = json.loads(archive.read(entry))
        shapes = annotation.get('shapes', [])
        shape_labels = [shape.get('label', '') for shape in shapes]

        names.append(pathlib.Path(entry).stem)
        labels_col.append(','.join(shape_labels))
        counts.append(sum(1 for label in shape_labels if label.lower() != 'finger'))

df = pd.DataFrame({'name': names, 'labels': labels_col, 'real_count': counts})
df.to_csv(prepared_labels, index=False)

print(f'wrote {len(df)} rows to {prepared_labels}')
df.head()

# Dataset statistics
Summarize the labels CSV into a **metadata** YAML: how many coins per image
(COUNTING) and how often each raw label occurs (CLASSIFICATION, keeping `finger` visible here
since it is a real, if rare, annotation — unlike `real_count`, this view is not meant as
ground truth for the counting task).

In [ ]:
labels_lists = df['labels'].str.split(',')
n_per_image = labels_lists.map(len)

counts_distribution = n_per_image.value_counts().sort_index()
label_distribution_counts = labels_lists.explode().value_counts()
label_distribution_percent = labels_lists.explode().value_counts(normalize=True)

print('counts distribution (labels per image -> number of images):')
print(counts_distribution.to_string())
print()
print('label distribution:')
print(label_distribution_counts.to_string())

In [ ]:
# Counts histogram
plt.figure(figsize=(7, 4))
plt.bar(counts_distribution.index.astype(str), counts_distribution.values)
plt.title('Counts Histogram (labels per image)')
plt.xlabel('Number of labels')
plt.ylabel('Frequency (images)')
plt.show()

# Label histogram
plt.figure(figsize=(7, 4))
plt.bar(label_distribution_counts.index, label_distribution_counts.values)
plt.title('Label Histogram (frequency of each label)')
plt.xlabel('Label')
plt.ylabel('Frequency (total)')
plt.show()

In [ ]:
metadata = {
    'counting': {
        'total_instances': int(len(df)),
        'counts': {
            'min': float(n_per_image.min()),
            'max': float(n_per_image.max()),
            'mean': float(n_per_image.mean()),
            'median': float(n_per_image.median()),
            'std': float(n_per_image.std()),
        },
        'counts_distribution': {int(k): int(v) for k, v in counts_distribution.items()},
    },
    'classification': {
        'label_distribution_counts': {str(k): int(v) for k, v in label_distribution_counts.items()},
        'label_distribution_percent': {str(k): float(v) for k, v in label_distribution_percent.items()},
        'label_mode': str(label_distribution_counts.idxmax()),
    },
}

with open(prepared_metadata, 'w', encoding='utf-8') as f:
    yaml.safe_dump(metadata, f, sort_keys=False, allow_unicode=True)

print(f'wrote {prepared_metadata}')

# Reading metadata.yaml

In [ ]:
with open(prepared_metadata, 'r', encoding='utf-8') as f:
    reloaded = yaml.safe_load(f)

print(yaml.safe_dump(reloaded, sort_keys=False, allow_unicode=True))

# Testing

In [ ]:
with zipfile.ZipFile(prepared_images, 'r') as archive:
    sample_name = sorted(archive.namelist())[0]
    sample_image = skimage.io.imread(io.BytesIO(archive.read(sample_name)))

plt.imshow(sample_image, cmap='gray')
plt.title(sample_name)
plt.show()